In [ ]:
%load_ext autoreload
%autoreload 2

# Imports

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from lineval.utils import (load_hub_docs,
                           check_assign_docs,
                           OpenAiWordSenseComparatorv4,
                           ClusterByMeaningModelv4,
                           annotator_filter,
                           AnnotationSerializerv2,
                           AnnotationFactoryv2,
                           get_unique_senses,
                           plot_cluster_accuracy
)
from linalgo.annotate.serializers import DocumentSerializer
from linalgo.annotate.models import Document, Annotation, Annotator

from linpub.metrics import accuracy

# Getting data

In [ ]:
X_docs = load_hub_docs()

In [ ]:
plot_cluster_accuracy(X_docs)

In [ ]:
gold_annotator = Annotator(name="gold",
                           owner="jack",
                           model="MACHINE")
gold_annotator.__dict__
gold_id = gold_annotator.id

In [ ]:
jack_id = 'd34602e1-1664-42cb-b139-c5cb8bcfa2a0'

# ONLY RUN ONCE!!!

for doc in X_docs:
    gold_annos = [anno for anno in doc.annotations if anno.annotator.id == jack_id]
    for anno in gold_annos:
        a = Annotation(entity = anno.entity,
                       body=anno.body,
                       document=doc,
                       annotator = gold_annotator,
                       target = anno.target,
                       )
        doc.annotations.add(a)

# NOT FOR NOW, ONLY IF WE WANT ML ANNOTATIONS ON BQ
# from linpub.elt.tasks.bq import save_docs_to_bq, save_annos_to_bq
# save_docs_to_bq(X_docs)
# save_annos_to_bq([anno for doc in X_docs for anno in doc.annotations])

In [ ]:
plot_cluster_accuracy(X_docs)

# Filtering data/errors if no gold

In [ ]:
#Don't run this, only if you want to send docs to Arnaud on the hub
# er_fl_docs = error_filter(X_docs, keep_correct= True, keep_errors= False)
# check_assign_docs(er_fl_docs)

# Predicting

In [ ]:
load_dotenv()
api_key = os.getenv("OPEN_AI_API_KEY")
comp_1 = OpenAiWordSenseComparatorv4(api_key=api_key, openai_model="gpt-4o-mini", thought_process=False)
model_1 = ClusterByMeaningModelv4(word_sense_comparator=comp_1)

In [ ]:
#testing with only 3 docs
X_small = X_gold[:3]
y = get_unique_senses(X_small)
y_pred_docs = model_1.predict(X_small)
y_pred = get_unique_senses(y_pred_docs)
accuracy(y_pred, y),len(y_pred_docs)

In [ ]:
list(y_pred_docs[0].annotations)#[0].__dict__

In [ ]:
len(X_docs), X_docs[0].__dict__, list(X_docs[0].annotations)[3].__dict__

In [ ]:
# X_gold = annotator_filter(X_docs, gold_id)
# len(X_gold), len(X_gold[0].annotations)

In [ ]:
y = get_unique_senses(X_docs)
y_pred_docs = model_1.predict(X_docs)
y_pred = get_unique_senses(y_pred_docs)

In [ ]:
plot_cluster_accuracy(X_docs)

# saving experiments

## experimenting

In [ ]:
X_docs = load_hub_docs()

In [ ]:
annotations = [ano for doc in X_docs for ano in list(doc.annotations) ]
lst = AnnotationSerializerv2(annotations).serialize()
len(lst), lst[0]

In [ ]:
df = pd.DataFrame(lst)
df.to_csv("annotations.csv", index=False)

In [ ]:
df = pd.read_csv("annotations.csv")
lst = df.to_dict(orient="records")
len(lst)

In [ ]:
rebuild_annotations = [AnnotationFactoryv2.from_dict(l) for l in lst]
# rebuild_annotations == annotations

In [ ]:
rebuild_annotations[0].document

In [ ]:
docs_lst = DocumentSerializer(X_docs).serialize()
df = pd.DataFrame(docs_lst)
df.to_csv("docs.csv", index=False)


In [ ]:

df = pd.read_csv("docs.csv")
lst = df.to_dict(orient="records")
rebuilt_docs = []
for l in lst:
    doc = Document(**l)
    rebuilt_docs.append(doc)

In [ ]:
rebuilt_docs[0].uri, rebuilt_docs[0].annotations[0].__dict__

In [ ]:
for doc in rebuilt_docs:
    doc_annos = []
    for anno in rebuild_annotations:
        if doc.uri == anno.document.uri:
            doc_annos.append(anno)
    doc.annotations = doc_annos

In [ ]:
rebuilt_docs == X_docs, rebuilt_docs[0]== X_docs[0], list(rebuilt_docs[0].annotations)[0] == list(X_docs[0].annotations)[0]

## save/load functions

In [ ]:
def save_docs(docs, filename):
    annos = [ano for doc in docs for ano in list(doc.annotations)]
    anno_lst = AnnotationSerializerv2(annos).serialize()
    df = pd.DataFrame(anno_lst)
    df.to_csv(filename+"annos", index=False)
    docs_lst = DocumentSerializer(docs).serialize()
    df = pd.DataFrame(docs_lst)
    df.to_csv(filename+"docs", index=False)

save_docs(X_docs, "test")


In [ ]:
def load_docs(filename):
    df = pd.read_csv(filename+"annos")
    lst = df.to_dict(orient="records")
    rebuild_annotations = [AnnotationFactoryv2.from_dict(l) for l in lst]
    df = pd.read_csv(filename+"docs")
    lst = df.to_dict(orient="records")
    rebuilt_docs = []
    for l in lst:
        doc = Document(**l)
        rebuilt_docs.append(doc)
    for doc in rebuilt_docs:
        doc_annos = []
        for anno in rebuild_annotations:
            if doc.uri == anno.document.uri:
                doc_annos.append(anno)
        doc.annotations = set(doc_annos)
    return rebuilt_docs

rebuilt_docs = load_docs("test")

In [ ]:
list(rebuilt_docs[0].annotations)[0].__dict__

In [ ]:
rebuilt_docs[0].__dict__

In [ ]:
list(X_docs[0].annotations)[0].__dict__

In [ ]:
X_docs[0].__dict__

# testing body serial/deserial

In [ ]:
from lineval.other_stuff import Body, BodySerializer, BodyFactory

In [ ]:
Body1 = Body(text = "This is a test")
Body2 = Body1
Body2.proba = 1.0
Body2.process = "ref"
Body3 = Body2
Body3.text = "This is a test 2"
Body3.proba = 0.5
Body3.process = "plop"
Body3.context = "ref"
Body3.text = "This is a test 3"
Body4 = "not a Body class"
Body1, Body2, Body3, Body4

In [ ]:
ser1 = BodySerializer().serialize(Body1)
ser2 = BodySerializer().serialize(Body2)
ser3 = BodySerializer().serialize(Body3)
ser4 = BodySerializer().serialize(Body4)
ser1, ser2, ser3, ser4

In [ ]:
fact = BodyFactory()
res1 = fact.deserialize(ser1)
res2 = fact.deserialize(ser2)
res3 = fact.deserialize(ser3)
res4 = fact.deserialize(ser4)
res1, res2, res3, res4

In [ ]:
res1 == Body1, res2 == Body2, res3 == Body3, res4 == Body4

# testing ano serial/factory

In [ ]:
from lineval.other_stuff import AnnotationSerializerv2, AnnotationFactoryv2

In [ ]:
from lineval.utils import load_hub_docs
X_docs = load_hub_docs()


In [ ]:
ano = list(X_docs[0].annotations)[0]
lst = AnnotationSerializerv2([ano]).serialize()
res = AnnotationFactoryv2.from_dict(lst[0])
res == ano

In [ ]:
annos = [ano for doc in X_docs for ano in list(doc.annotations)]
lst = AnnotationSerializerv2(annos).serialize()
rebuilt_annos = [AnnotationFactoryv2.from_dict(l) for l in lst]
len(rebuilt_annos), annos == rebuilt_annos

In [ ]:
rebuilt_docs = set([anno.document for anno in annos])
len(rebuilt_docs), len(X_docs)

In [ ]:
mismatch = False
for doc in X_docs:
    for r_doc in rebuilt_docs:
        if doc.uri == r_doc.uri and doc != r_doc:
            print(doc.uri)
            mismatch = True
if not mismatch:
    print("All good")
